# Stage 1 - WP3: Relation Correction and Review Preparation

| **Work package** | WP3 |
|---|---|
| **Task** | Correct the WP2 relations before building the knowledge graph |
| **Status** | Candidate correction pipeline - review sample still needs human review |

The official WP2 handover (`outputs/stage1_wp2_open_models_official_20261007/`) is used **as is**: no WP2 file is modified. This notebook adds a correction
layer on top of it:

```
WP2 handover (unchanged)
  -> correction tables (small, readable, editable)
  -> corrected nodes / evidence / edges (every change has a reason)
  -> tiered edges for the knowledge graph + a fresh human-review sample
```

**Why a correction layer?** A manual review of WP2 relations (90 sampled triples) found that most errors come from
the entity stage rather than from GLiREL itself:
- *entity linking*: the WP1 regex for Llama also matches "Llama Nemotron", "Llama Stack" and "LlamaCon", so NVIDIA
  products become `Llama -> developed by -> NVIDIA` (42 sentences);
- *aliases and variants*: the same organisation exists as several nodes (`Ai2` / `Allen Institute for AI`,
  `MCP` / `Model Context Protocol`, `OpenAIâ`, `NVIDIA는`);
- *wrong node types*: `Alibaba` is a place, `TII` is a model;
- *generic nodes*: `model`, `company`, `AI models`, `GPUs` take part in relations.

Instead of a score threshold (which removed many correct relations together with the wrong ones), each error source
gets its own rule. Relations are never deleted: each edge gets a `tier` and, if excluded, an `exclusion_reason`.

| tier | meaning |
|---|---|
| `A` | curated model -> developer from the WP1 lexicon |
| `B` | GLiREL relation that passes all checks and is stated in >= 2 distinct sentences |
| `C` | passes all checks but only one distinct sentence |
| `X` | excluded (reason in `exclusion_reason`) |

## Setup

In [1]:
import hashlib, json, os, re, unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Repository root: local project checkout or Colab clone.
CANDIDATES = [Path.cwd(), *Path.cwd().parents, Path('/content/computational-language-technology-hs26')]
WP2_RELATIVE = Path('outputs/stage1_wp2_open_models_official_20261007')
REPO_DIR = next(p for p in CANDIDATES if (p / WP2_RELATIVE / 'edges_relations.csv').exists())
WP2_DIR = REPO_DIR / WP2_RELATIVE
LEXICON_PATH = REPO_DIR / 'artifacts/stage1_wp1/lexicon/open_models_lexicon.json'
OUT_DIR = REPO_DIR / 'outputs/stage1_wp3_relation_correction_20261008'
EVAL_DIR = REPO_DIR / 'artifacts/stage1_wp3/evaluation/relation_correction'
(OUT_DIR / 'corrections').mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Earlier review samples: their triples are excluded from the new sample (paths separated by os.pathsep)
PRIOR_SAMPLE_FILES = [Path(p) for p in os.environ.get('WP3_PRIOR_SAMPLES', '').split(os.pathsep) if p]
PRIOR_SAMPLE_FILES += sorted((REPO_DIR / 'artifacts/stage1_wp2/evaluation').glob('**/relation_review_*seed*.csv'))

SEED = 2026
SAMPLE_SIZE = 90
print(f"repository: {REPO_DIR.name}")

repository: 2026-09-CLT-computational_language


In [2]:
nodes = pd.read_csv(WP2_DIR / 'nodes.csv', keep_default_na=False, na_values=[''])
edges = pd.read_csv(WP2_DIR / 'edges_relations.csv')
evidence = pd.read_csv(WP2_DIR / 'relations_evidence.csv')
sentences = pd.read_csv(WP2_DIR / 'sentences.csv')
schema = json.loads((WP2_DIR / 'schema.json').read_text(encoding='utf-8'))
lexicon = json.loads(LEXICON_PATH.read_text(encoding='utf-8'))

evidence = evidence[evidence['source'] == 'glirel'].copy()
evidence['sentence_id'] = evidence['sentence_id'].astype(int)
lexicon_edges = edges[edges['source'] == 'wp1_lexicon'].copy()
REL_TYPES = schema['edges']['relation']['types']

print(f"nodes: {len(nodes):,} | GLiREL evidence rows: {len(evidence):,} | "
      f"GLiREL triples: {(edges['source'] == 'glirel').sum():,} | lexicon edges: {len(lexicon_edges)}")

nodes: 54,597 | GLiREL evidence rows: 30,560 | GLiREL triples: 22,957 | lexicon edges: 35


## 1. Node corrections
### 1a. Name variants (automatic)
Mentions that differ only by casing, punctuation, a mojibake tail (`OpenAIâ`) or a Korean particle attached to a
Latin name (`NVIDIA는`) are merged into the most frequent node, but only within the same type group
(models with models, organisations with organisations), so that e.g. the company *Mistral* is not merged into a place.

In [3]:
TYPE_GROUP = {'FOUNDATION_MODEL': 'model', 'AI_MODEL': 'model', 'COMPANY': 'org', 'ORG': 'org'}

def name_key(name):
  s = unicodedata.normalize('NFKC', str(name))
  s = re.sub(r'â\S*$', '', s)                      # mojibake tail of a curly quote: OpenAIâ -> OpenAI
  s = re.sub(r'(?<=[A-Za-z0-9])[가-힣]+$', '', s)  # Korean particle after a Latin name: NVIDIA는
  return re.sub(r'[^0-9a-z]+', ' ', s.lower()).strip()

nodes['key'] = nodes['entity'].map(name_key)
nodes['group'] = nodes['entity_type'].map(TYPE_GROUP).fillna(nodes['entity_type'])
nodes['is_lexicon'] = nodes['canonical_id'].notna()

variant_rows = []
for (key, group), g in nodes[nodes['key'] != ''].groupby(['key', 'group']):
  if len(g) < 2:
    continue
  keep = g.sort_values(['is_lexicon', 'mentions'], ascending=False).iloc[0]
  for _, r in g[g['entity_id'] != keep['entity_id']].iterrows():
    variant_rows.append({'from_id': r['entity_id'], 'from_name': r['entity'], 'to_id': keep['entity_id'],
                         'to_name': keep['entity'], 'reason': 'name_variant'})
name_variants = pd.DataFrame(variant_rows)
print(f"name variants merged: {len(name_variants):,}")
display(name_variants.merge(nodes[['entity_id', 'mentions']], left_on='from_id', right_on='entity_id')
                     .sort_values('mentions', ascending=False).head(12)[['from_name', 'to_name', 'to_id', 'mentions']])

name variants merged: 375


,from_name,to_name,to_id,mentions
207,llama,Llama,model:llama,24
356,vision language models,vision-language models,model:vision_language_models_2,23
77,closed-source,closed source,license:closed_source,14
138,gemma,Gemma,model:gemma,14
252,open source community,open-source community,org:open_source_community_2,13
188,instruction-following,instruction following,capability:instruction_following,13
266,open-source software,open source software,license:open_source_software,12
294,Retrieval Augmented Generation,retrieval-augmented generation,technology:retrieval_augmented_generation_2,12
161,Grouped Query Attention,Grouped-Query Attention,architecture:grouped_query_attention_2,10
192,ip adapter,IP-Adapter,hardware:ip_adapter,10


### 1b. Curated aliases, node overrides and developer equivalents
These tables are the part that needs human judgement. They are seeded from the manual review and from the WP1
lexicon developers (several lexicon developers point to a wrongly typed or duplicate node, e.g. Qwen -> `place:alibaba`,
Falcon -> `model:tii`). Add rows here when the review finds new cases.

- `ALIAS_MAP`: two ids are **the same entity** -> merged
- `NODE_OVERRIDES`: fix the name or type of a (merged) node; a new id can be created for a corrected type
- `DEVELOPER_EQUIVALENTS`: organisations that are **not** the same entity but are acceptable as the developer of a
  family (e.g. Google DeepMind for Gemma). Used only for the consistency check, never merged.

In [4]:
ALIAS_MAP = [  # from_id, to_id, note
  ('org:deepseek_2', 'org:deepseek', 'lexicon developer node duplicates the DeepSeek company'),
  ('place:alibaba', 'org:alibaba', 'Alibaba typed as a place; company node'),
  ('org:alibaba_group', 'org:alibaba', 'same company'),
  ('org:alibaba_group_holding_limited', 'org:alibaba', 'same company'),
  ('org:alibaba_groupshares', 'org:alibaba', 'tokenisation artefact'),
  ('model:tii', 'org:technology_innovation_institute', 'TII typed as a model'),
  ('model:stability_ai', 'org:stability_ai', 'Stability AI typed as a model'),
  ('org:stability', 'org:stability_ai', 'same company'),
  ('org:moonshot', 'org:moonshot_ai', 'Moonshot = MoonShot AI'),
  ('org:z_ai', 'org:zhipu_ai_z_ai', 'Z.ai = Zhipu AI'),
  ('org:allen_institute_for_ai', 'org:ai2', 'Ai2 = Allen Institute for AI'),
  ('org:allen_institute_for_artificial_intelligence', 'org:ai2', 'Ai2 = Allen Institute for AI'),
  ('technology:model_context_protocol', 'standard:mcp', 'MCP = Model Context Protocol'),
  ('org:fundamental_artificial_intelligence_research_unit', 'org:fair', 'FAIR = Fundamental AI Research'),
  ('org:nous', 'org:nous_research', 'same company'),
  ('org:nousresearch', 'org:nous_research', 'same company'),
  ('org:cohere_for_ai', 'org:cohere_labs', 'Cohere For AI was renamed Cohere Labs'),
  ('org:cohere_lab', 'org:cohere_labs', 'same lab'),
  ('org:black_forest_labs', 'org:black_forest_labs_2', 'same company (lexicon developer node)'),
  ('person:anthropic_claude', 'model:claude', 'Claude typed as a person'),
  ('platform:meta_platforms', 'org:meta', 'Meta Platforms (legal name) typed as a platform'),
  ('org:meta_platforms_inc', 'org:meta', 'legal name of Meta'),
  ('org:meta_platforms_inc_2', 'org:meta', 'legal name of Meta'),
  ('org:meta_platforms_inc_3', 'org:meta', 'legal name of Meta'),
  ('org:deepmind', 'org:google_deepmind', 'DeepMind was renamed Google DeepMind in 2023'),
  ('org:deep_seek', 'org:deepseek', 'spelling variant'),
  ('org:google_llc', 'org:google', 'legal name of Google'),
  ('org:alphabet_inc', 'org:alphabet', 'legal name of Alphabet'),
  ('org:alibaba_group_holding', 'org:alibaba', 'same company'),
  ('org:nvidia_co', 'org:nvidia', 'same company'),
  ('org:cognition_ai', 'org:cognition', 'same company'),
  ('org:perplexity_ai', 'org:perplexity', 'same company'),
  ('org:rand_corporation', 'org:rand', 'same organisation'),
  ('org:red_hat_inc_2', 'org:red_hat', 'legal name of Red Hat'),
  ('org:redhat', 'org:red_hat', 'spelling variant'),
  ('org:liquid', 'org:liquid_ai', 'Liquid = Liquid AI'),
  ('org:ant', 'org:ant_group', 'Ant = Ant Group'),
  ('org:thinking_machines', 'org:thinking_machines_lab', 'Thinking Machines = Thinking Machines Lab'),
  ('org:counterpoint', 'org:counterpoint_research', 'Counterpoint = Counterpoint Research'),
  ('org:hugging_face', 'technology:hugging_face', 'lexicon developer node duplicates Hugging Face'),
]
NODE_OVERRIDES = [  # entity_id, entity, entity_type, note
  ('org:alibaba', 'Alibaba', 'COMPANY', 'node existed only as the variant "Alibabaâ"'),
  ('org:stability_ai', 'Stability AI', 'COMPANY', 'new id for the corrected type'),
  ('org:technology_innovation_institute', 'Technology Innovation Institute (TII)', 'ORG', 'lexicon developer of Falcon'),
  ('org:black_forest_labs_2', 'Black Forest Labs', 'COMPANY', ''),
  ('person:devin', 'Devin', 'AI_MODEL', 'Cognition coding agent, not a person (check)'),
  ('technology:hugging_face', 'Hugging Face', 'TECHNOLOGY', 'display name was the id-like "hugging_face"'),
]
DEVELOPER_EQUIVALENTS = {  # lexicon developer -> organisations also accepted as developer of its families
  'org:google': ['org:google_deepmind', 'org:deepmind', 'org:google_research'],
  'org:alibaba': ['platform:alibaba_cloud', 'org:alibaba_cloud_model_studio'],
  'org:meta': ['org:fair', 'model:meta_ai'],
  'org:nvidia': ['org:nvidia_research'],
  'org:cohere': ['org:cohere_labs'],
  'org:g42_inception': ['org:g42', 'org:inception'],
  'org:mistral_ai': ['model:mistral'],
  'org:minimax': ['product:minimax'],
}

In [5]:
# Resolve every original id to its final id (name variants first, then curated aliases; chains are followed)
merge_map = dict(zip(name_variants['from_id'], name_variants['to_id']))
merge_map.update({a: b for a, b, _ in ALIAS_MAP})

def resolve(eid):
  seen = set()
  while eid in merge_map and eid not in seen:
    seen.add(eid)
    eid = merge_map[eid]
  return eid

nodes['final_id'] = nodes['entity_id'].map(resolve)

# Corrected node table: one row per final id, counts summed over the merged ids
agg = (nodes.groupby('final_id')
            .agg(merged_ids=('entity_id', lambda x: ' | '.join(sorted(x))), mentions=('mentions', 'sum'),
                 n_docs=('n_docs', 'max'), first_seen=('first_seen', 'min'), last_seen=('last_seen', 'max'))
            .reset_index().rename(columns={'final_id': 'entity_id'}))
base = nodes.set_index('entity_id')[['entity', 'entity_type', 'canonical_id', 'developer', 'model_versions']]
nodes_corr = agg.join(base, on='entity_id')
for eid, name, etype, note in NODE_OVERRIDES:
  if eid not in set(nodes_corr['entity_id']):
    nodes_corr = pd.concat([nodes_corr, pd.DataFrame([{'entity_id': eid, 'merged_ids': eid, 'mentions': 0}])],
                           ignore_index=True)
  nodes_corr.loc[nodes_corr['entity_id'] == eid, ['entity', 'entity_type']] = [name, etype]
nodes_corr['n_merged'] = nodes_corr['merged_ids'].str.count(r'\|') + 1
print(f"nodes: {len(nodes):,} -> {len(nodes_corr):,} after merging")

nodes: 54,597 -> 54,188 after merging


### 1c. Generic nodes (rule-based)
For **named-entity types** (models, organisations, people, products, platforms, hardware, technologies, ...), a node
is *generic* when its name contains no proper-name token, e.g. `model`, `company`, `AI models`, `GPUs`, `Big Tech`,
`22 datasets`. A token counts as a proper name when it contains an upper-case letter (and is not a class word such as
AI, LLM, GPU) or mixes letters and digits (`o3`, `R1`). **Concept types** (`CAPABILITY`, `ARCHITECTURE`, `MODEL_SPEC`,
`OPENNESS_CONCEPT`) are lower-case by nature (*reasoning*, *mixture of experts*), so only vague words such as
*capabilities* or *frameworks* are generic there. Lexicon entities and named WP1 normalisations (`united_states`,
`hugging_face`) are never generic; `NOT_GENERIC` lists lower-case brand names found during the check (`watsonx`).
The rule replaces the word list used before, so it also covers names that have not been seen in a review yet.
The flagged list is exported for a quick manual check.

In [6]:
CLASS_WORDS = {'ai', 'genai', 'llm', 'llms', 'slm', 'slms', 'vlm', 'vlms', 'ml', 'gpu', 'gpus', 'cpu', 'cpus',
               'vm', 'vms', 'api', 'apis', 'app', 'apps', 'big', 'tech', 'ceo', 'cto', 'cfo', 'coo', 'vl', 'us',
               'rl', 'rlhf', 'rag', 'nlp', 'ocr', 'vr', 'ar', 'xr',                       # technique acronyms
               'the', 'this', 'these', 'those', 'its', 'our', 'their', 'other', 'new'}   # determiners
CONCEPT_TYPES = {'CAPABILITY', 'ARCHITECTURE', 'MODEL_SPEC', 'OPENNESS_CONCEPT'}
VAGUE_WORDS = {'capability', 'capabilities', 'ability', 'abilities', 'feature', 'features', 'framework', 'frameworks',
               'architecture', 'architectures', 'model', 'models', 'technology', 'technologies', 'performance'}
NOT_GENERIC = {'watsonx'}
NORMS = {norm for _, norm in lexicon['phrase_normalisation']}
CLASS_NORMS = {'llm', 'slm', 'ai', 'generative_ai', 'foundation_model', 'reasoning_model', 'open_source', 'open_weight'}

def is_generic(name, entity_type, is_lexicon):
  name = str(name)
  if is_lexicon or name.lower() in NOT_GENERIC:
    return False
  if entity_type in CONCEPT_TYPES:
    words = name.lower().split()
    return bool(words) and all(w in VAGUE_WORDS for w in words)
  if name in NORMS:
    return name in CLASS_NORMS
  tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9.+'\-]*", name)
  if not tokens:  # non-Latin names are kept
    return False
  def proper(t):
    if re.search(r'[A-Za-z]', t) and re.search(r'\d', t):
      return True
    return any(c.isupper() for c in t) and t.lower() not in CLASS_WORDS and t.lower().rstrip('s') not in CLASS_WORDS
  return not any(proper(t) for t in tokens)

nodes_corr['is_generic'] = [is_generic(n, t, pd.notna(c)) for n, t, c in
                            zip(nodes_corr['entity'], nodes_corr['entity_type'], nodes_corr['canonical_id'])]
print(f"generic nodes: {nodes_corr['is_generic'].sum():,} of {len(nodes_corr):,}")
display(nodes_corr[nodes_corr['is_generic']].nlargest(20, 'mentions')[['entity_id', 'entity', 'entity_type', 'mentions']])

generic nodes: 20,798 of 54,188


,entity_id,entity,entity_type,mentions
49321,technology:ai,AI,TECHNOLOGY,7631
23771,model:model,model,AI_MODEL,3283
23553,model:llm,llm,AI_MODEL,2404
13392,dataset:data,data,DATASET,970
28546,org:company,company,COMPANY,893
36502,person:developers,developers,PERSON,821
23795,model:models,models,AI_MODEL,763
49542,technology:api,API,TECHNOLOGY,758
20925,license:open_source,open_source,LICENSE,657
18517,hardware:gpu,GPU,HARDWARE,602


## 2. Evidence corrections (one row per sentence)
1. head and tail are mapped to the corrected nodes; relations that become self-relations are excluded;
2. **family span**: a lexicon family is dropped from a sentence when it only occurs inside another product name
   (`Llama Nemotron`, `Llama Stack`, `LlamaCon`). The lexicon regex is re-applied after removing these names;
3. `competes with` is symmetric: head and tail are put in a fixed order so both directions count as one relation;
4. type constraints are re-checked with the corrected types (some relations become valid, e.g. `Qwen -> Alibaba`).

In [7]:
MATCH_MAP = str.maketrans({"‐": "-", "‑": "-", "‒": "-", "–": "-", "−": "-", "‘": "'", "’": "'", "“": '"', "”": '"'})
FAMILY_RX = {f'model:{fid}': re.compile(f['rx']) for fid, f in lexicon['families'].items()}
FAMILY_SPAN_EXCEPTIONS = {  # family node -> names that contain the family name but are other products
  'model:llama': re.compile(r"\bLlama[\w.\- ]{0,15}?(?:Nemotron|Stack)\b|\bLlamaCon\b"),
}
SYMMETRIC = {'competes with'}
node_info = nodes_corr.set_index('entity_id')

ev = evidence.copy()
ev['orig_head_id'], ev['orig_tail_id'] = ev['head_id'], ev['tail_id']
ev['head_id'] = ev['head_id'].map(resolve)
ev['tail_id'] = ev['tail_id'].map(resolve)
ev['evidence_status'] = 'ok'

ev.loc[ev['head_id'] == ev['tail_id'], 'evidence_status'] = 'self_relation_after_merge'

def family_span_only(family_id, text):
  rx_ex = FAMILY_SPAN_EXCEPTIONS.get(family_id)
  if rx_ex is None:
    return False
  view = str(text).translate(MATCH_MAP)
  return bool(rx_ex.search(view)) and not FAMILY_RX[family_id].search(rx_ex.sub(' ', view))

for side in ('head_id', 'tail_id'):
  mask = ev[side].isin(FAMILY_SPAN_EXCEPTIONS) & (ev['evidence_status'] == 'ok')
  hit = [family_span_only(f, t) for f, t in zip(ev.loc[mask, side], ev.loc[mask, 'sentence_text'])]
  ev.loc[mask[mask].index[hit], 'evidence_status'] = 'family_span'

swap = ev['relation'].isin(SYMMETRIC) & (ev['head_id'] > ev['tail_id'])
ev.loc[swap, ['head_id', 'tail_id', 'orig_head_id', 'orig_tail_id']] = \
    ev.loc[swap, ['tail_id', 'head_id', 'orig_tail_id', 'orig_head_id']].values

ev['head'] = ev['head_id'].map(node_info['entity'])
ev['tail'] = ev['tail_id'].map(node_info['entity'])
ev['head_type'] = ev['head_id'].map(node_info['entity_type'])
ev['tail_type'] = ev['tail_id'].map(node_info['entity_type'])

def type_ok(rel, h, t):
  spec = REL_TYPES.get(rel)
  if spec is None:
    return False
  if rel in SYMMETRIC:
    return (h in spec['head_types'] and t in spec['tail_types']) or (t in spec['head_types'] and h in spec['tail_types'])
  return h in spec['head_types'] and t in spec['tail_types']

ev['type_valid_corrected'] = [type_ok(r, h, t) for r, h, t in zip(ev['relation'], ev['head_type'], ev['tail_type'])]
ev['sentence_key'] = ev['sentence_text'].fillna('').map(lambda s: re.sub(r'\W+', ' ', s.lower()).strip())

print(ev['evidence_status'].value_counts().to_string())
print(f"\ntype-valid evidence rows: WP2 {evidence['type_valid'].sum():,} -> corrected {ev['type_valid_corrected'].sum():,}")

evidence_status
ok                           30448
family_span                    110
self_relation_after_merge        2

type-valid evidence rows: WP2 19,613 -> corrected 19,454


## 3. Edge aggregation and relation checks
Evidence rows with status `ok` are aggregated per `(head_id, relation, tail_id)`. Support is counted on **distinct
sentence texts**, since syndicated articles repeat the same sentence. Then the triple-level checks are applied:

| check | rule |
|---|---|
| `type_invalid` | head/tail types do not satisfy the schema after correction |
| `generic_node` | head or tail is a generic node |
| `conflicts_lexicon_developer` | `developed/released/announced by` from a lexicon family to an organisation that is not its WP1 developer (or an accepted equivalent) |
| `own_product_or_parent` | `competes with` / `outperforms` between an artefact and its own developer or parent company |
| `reverse_duplicate` | the same directional relation exists in both directions: the direction with less support is excluded |

In [8]:
ok = ev[ev['evidence_status'] == 'ok']
edges_corr = (ok.groupby(['head_id', 'relation', 'tail_id'])
                .agg(head=('head', 'first'), head_type=('head_type', 'first'), tail=('tail', 'first'),
                     tail_type=('tail_type', 'first'), type_valid=('type_valid_corrected', 'first'),
                     n_sentences=('sentence_key', 'size'), n_distinct_sentences=('sentence_key', 'nunique'),
                     n_docs=('doc_id', 'nunique'), mean_score=('score', 'mean'),
                     first_seen=('date', 'min'), last_seen=('date', 'max'),
                     orig_head_ids=('orig_head_id', lambda x: ' | '.join(sorted(set(x)))),
                     orig_tail_ids=('orig_tail_id', lambda x: ' | '.join(sorted(set(x)))))
                .reset_index())
edges_corr['source'] = 'glirel'
edges_corr['mean_score'] = edges_corr['mean_score'].round(3)
reasons = {i: [] for i in edges_corr.index}
def flag(mask, reason):
  for i in edges_corr.index[mask]:
    reasons[i].append(reason)

generic_ids = set(nodes_corr.loc[nodes_corr['is_generic'], 'entity_id'])
flag(~edges_corr['type_valid'], 'type_invalid')
flag(edges_corr['head_id'].isin(generic_ids) | edges_corr['tail_id'].isin(generic_ids), 'generic_node')

# Lexicon developers (after alias resolution) and accepted equivalents
family_dev = {resolve(h): resolve(t) for h, t in zip(lexicon_edges['head_id'], lexicon_edges['tail_id'])}
accepted_dev = {f: {d, *map(resolve, DEVELOPER_EQUIVALENTS.get(d, []))} for f, d in family_dev.items()}
DEV_RELATIONS = {'developed by', 'released by', 'announced by'}
conflict = [r in DEV_RELATIONS and h in accepted_dev and t not in accepted_dev[h]
            for h, r, t in zip(edges_corr['head_id'], edges_corr['relation'], edges_corr['tail_id'])]
flag(np.array(conflict), 'conflicts_lexicon_developer')

# Ownership pairs: lexicon + the dominant developer/parent of each artefact among relations that passed the checks
# so far (>= 2 distinct sentences and at least half of the artefact's support), so that one wrong
# "o1 developed by DeepSeek" does not turn "DeepSeek outperforms o1" into an own-product relation
clean = edges_corr.index[[not reasons[i] for i in edges_corr.index]]
owned = edges_corr.loc[clean]
owned = owned[owned['relation'].isin(DEV_RELATIONS | {'subsidiary of'})]
owned = owned.groupby(['head_id', 'tail_id'])['n_distinct_sentences'].sum().reset_index()
owned['share'] = owned['n_distinct_sentences'] / owned.groupby('head_id')['n_distinct_sentences'].transform('sum')
owned = owned[(owned['n_distinct_sentences'] >= 2) & (owned['share'] >= 0.5)]
owner_pairs = {frozenset(p) for p in zip(owned['head_id'], owned['tail_id'])}
for f, devs in accepted_dev.items():
  owner_pairs |= {frozenset((f, d)) for d in devs}
own = [r in {'competes with', 'outperforms'} and frozenset((h, t)) in owner_pairs
       for h, r, t in zip(edges_corr['head_id'], edges_corr['relation'], edges_corr['tail_id'])]
flag(np.array(own), 'own_product_or_parent')

# Reverse duplicates of directional relations: keep the better-supported direction
key = {(h, r, t): i for i, (h, r, t) in enumerate(zip(edges_corr['head_id'], edges_corr['relation'], edges_corr['tail_id']))}
for (h, r, t), i in key.items():
  j = key.get((t, r, h))
  if r in SYMMETRIC or j is None or i > j:
    continue
  a, b = edges_corr.loc[i], edges_corr.loc[j]
  loser = j if (a['n_distinct_sentences'], a['mean_score']) >= (b['n_distinct_sentences'], b['mean_score']) else i
  reasons[loser].append('reverse_duplicate')

edges_corr['exclusion_reason'] = [';'.join(reasons[i]) for i in edges_corr.index]
edges_corr['tier'] = np.select([edges_corr['exclusion_reason'] != '', edges_corr['n_distinct_sentences'] >= 2],
                               ['X', 'B'], default='C')

In [9]:
# Tier A: curated lexicon edges (ids resolved; DeepSeek -> deepseek becomes a self-relation and is dropped)
lex = lexicon_edges.assign(head_id=lexicon_edges['head_id'].map(resolve), tail_id=lexicon_edges['tail_id'].map(resolve))
lex = lex[lex['head_id'] != lex['tail_id']]
lex = lex.assign(head=lex['head_id'].map(node_info['entity']), tail=lex['tail_id'].map(node_info['entity']),
                 head_type=lex['head_id'].map(node_info['entity_type']), tail_type=lex['tail_id'].map(node_info['entity_type']),
                 tier='A', exclusion_reason='', type_valid=True, orig_head_ids=lexicon_edges['head_id'],
                 orig_tail_ids=lexicon_edges['tail_id'])
edges_corr = pd.concat([lex[[c for c in edges_corr.columns if c in lex.columns]], edges_corr], ignore_index=True)

# Graph input: collapse exact A/B duplicates while preserving all source labels and GLiREL support.
CORE_TIERS = {'A', 'B'}
TRIPLE_KEY = ['head_id', 'relation', 'tail_id']
core = edges_corr[edges_corr['tier'].isin(CORE_TIERS)].copy()
core['_tier_order'] = core['tier'].map({'A': 0, 'B': 1})
graph_edges = (core.sort_values(TRIPLE_KEY + ['_tier_order'])
                   .drop_duplicates(TRIPLE_KEY, keep='first')
                   .drop(columns='_tier_order'))
core_support = (core.groupby(TRIPLE_KEY)
                    .agg(sources=('source', lambda x: ' | '.join(sorted(set(x)))),
                         n_sentences_graph=('n_sentences', 'max'),
                         n_distinct_sentences_graph=('n_distinct_sentences', 'max'),
                         n_docs_graph=('n_docs', 'max'),
                         mean_score_graph=('mean_score', 'max'))
                    .reset_index())
graph_edges = graph_edges.merge(core_support, on=TRIPLE_KEY, how='left', validate='one_to_one')
for original, combined in [('n_sentences', 'n_sentences_graph'),
                           ('n_distinct_sentences', 'n_distinct_sentences_graph'),
                           ('n_docs', 'n_docs_graph'), ('mean_score', 'mean_score_graph')]:
  graph_edges[original] = graph_edges[combined].combine_first(graph_edges[original])
  graph_edges = graph_edges.drop(columns=combined)
print(f"candidate graph edges: {len(core):,} rows -> {len(graph_edges):,} unique triples")

wp2_suggested = ((edges['source'] == 'glirel') & edges['type_valid'] & (edges['n_sentences'] >= 2)).sum()
print(f"WP2 suggested filter (type_valid & n_sentences >= 2): {wp2_suggested:,} triples")
print(edges_corr['tier'].value_counts().sort_index().to_string())
reason_counts = edges_corr['exclusion_reason'].str.split(';').explode().replace('', np.nan).dropna().value_counts()
print('\nexclusion reasons (an edge can have several):')
print(reason_counts.to_string())

candidate graph edges: 1,085 rows -> 1,070 unique triples
WP2 suggested filter (type_valid & n_sentences >= 2): 1,629 triples
tier
A       33
B     1052
C     7780
X    13507

exclusion reasons (an edge can have several):
exclusion_reason
type_invalid                   9146
generic_node                   6872
conflicts_lexicon_developer     435
reverse_duplicate               225
own_product_or_parent            93


### Example corrections

In [10]:
show = ['head', 'relation', 'tail', 'n_distinct_sentences', 'tier', 'exclusion_reason']
print('Largest excluded triples with at least two distinct sentences:')
display(edges_corr[(edges_corr['tier'] == 'X') & (edges_corr['n_distinct_sentences'] >= 2)]
          .nlargest(12, 'n_distinct_sentences')[show])
print('Lexicon families: developed by in tier B')
display(edges_corr[edges_corr['head_id'].isin(family_dev) & (edges_corr['relation'] == 'developed by') &
                   (edges_corr['tier'] == 'B')][show])
print('Evidence removed by the family-span rule (most frequent relations):')
display(ev[ev['evidence_status'] == 'family_span'].groupby(['head', 'relation', 'tail']).size()
          .sort_values(ascending=False).head(8))

Largest excluded triples with at least two distinct sentences:


,head,relation,tail,n_distinct_sentences,tier,exclusion_reason
20601,AI,developed by,NVIDIA,130.0,X,generic_node
16869,China,competes with,united_states,72.0,X,type_invalid
20526,AI,developed by,DeepSeek,62.0,X,generic_node
20549,AI,developed by,Google,52.0,X,generic_node
20604,AI,developed by,OpenAI,49.0,X,generic_node
1608,open_model,developed by,NVIDIA,46.0,X,type_invalid
13205,OpenAI,competes with,AI,40.0,X,generic_node
16885,China,competes with,AI,40.0,X,type_invalid;generic_node
20519,AI,developed by,company,38.0,X,generic_node
20583,AI,developed by,Meta,38.0,X,generic_node


Lexicon families: developed by in tier B


,head,relation,tail,n_distinct_sentences,tier,exclusion_reason
4307,Command R / Aya,developed by,Cohere,8.0,B,
4308,Command R / Aya,developed by,Cohere Labs,2.0,B,
4572,Falcon,developed by,Technology Innovation Institute (TII),3.0,B,
4775,Gemma,developed by,Google,75.0,B,
4776,Gemma,developed by,Google DeepMind,11.0,B,
5273,gpt-oss,developed by,OpenAI,26.0,B,
5318,Granite,developed by,IBM,35.0,B,
5358,Grok (open releases),developed by,xAI,2.0,B,
5371,Hermes,developed by,Nous Research,4.0,B,
5408,Hunyuan,developed by,Tencent,10.0,B,


Evidence removed by the family-span rule (most frequent relations):


head   relation         tail        
Llama  developed by     NVIDIA          30
                        VL              11
       supported by     OpenShift        4
VL     developed by     Llama            3
Llama  supported by     Qdrant           3
       available on     Hugging Face     3
       developed by     Meta             3
       integrated into  Red Hat          2
dtype: int64

## 4. Save the corrected tables

In [11]:
nodes_out = nodes_corr.drop(columns=[c for c in ('key', 'group') if c in nodes_corr.columns])
evidence_out = ev.drop(columns=['sentence_key'])
files = {
  'nodes_corrected.csv': nodes_out,
  'edges_relations_corrected.csv': edges_corr,
  'relations_evidence_corrected.csv': evidence_out,
  'edges_graph_candidates.csv': graph_edges,
  'corrections/name_variants.csv': name_variants,
  'corrections/alias_map.csv': pd.DataFrame(ALIAS_MAP, columns=['from_id', 'to_id', 'note']),
  'corrections/node_overrides.csv': pd.DataFrame(NODE_OVERRIDES, columns=['entity_id', 'entity', 'entity_type', 'note']),
  'corrections/developer_equivalents.csv': pd.DataFrame([(d, e) for d, es in DEVELOPER_EQUIVALENTS.items() for e in es],
                                                        columns=['lexicon_developer_id', 'accepted_id']),
  'corrections/generic_nodes.csv': nodes_out[nodes_out['is_generic']][['entity_id', 'entity', 'entity_type', 'mentions']]
                                     .sort_values('mentions', ascending=False),
}
for name, table in files.items():
  table.to_csv(OUT_DIR / name, index=False)
  print(f"{name:42} {len(table):>8,} rows")

nodes_corrected.csv                          54,188 rows
edges_relations_corrected.csv                22,372 rows


relations_evidence_corrected.csv             30,560 rows
edges_graph_candidates.csv                    1,070 rows
corrections/name_variants.csv                   375 rows
corrections/alias_map.csv                        40 rows
corrections/node_overrides.csv                    6 rows
corrections/developer_equivalents.csv            13 rows
corrections/generic_nodes.csv                20,798 rows


## 5. Fresh review sample of tier B
The corrections were designed from an earlier review, so their effect has to be measured on **new** triples:
- population: tier B GLiREL triples, excluding every triple of the earlier review samples (after id resolution);
- 90 triples, stratified by relation type with proportional allocation (largest remainder), so that the sample
  precision is an estimate for the whole of tier B;
- one primary evidence sentence (random among distinct sentences) with the sentences before and after, plus up to two
  other distinct supporting sentences.

The output has the same columns as the earlier review files, plus `tier`, `correction_notes` and the original ids.

In [12]:
rng = np.random.default_rng(SEED)

prior = set()
for path in PRIOR_SAMPLE_FILES:
  if path.exists():
    p = pd.read_csv(path)
    for h, r, t in zip(p['head_id'], p['predicate'], p['tail_id']):
      h, t = resolve(h), resolve(t)
      prior |= {(h, r, t), (t, r, h)}
print(f"earlier sample files: {sum(p.exists() for p in PRIOR_SAMPLE_FILES)} | excluded triples: {len(prior) // 2}")

pop = edges_corr[(edges_corr['tier'] == 'B') & (edges_corr['source'] == 'glirel')].copy()
pop = pop[[(h, r, t) not in prior for h, r, t in zip(pop['head_id'], pop['relation'], pop['tail_id'])]]

# Proportional allocation by relation type (largest remainder)
share = pop['relation'].value_counts() / len(pop) * SAMPLE_SIZE
alloc = np.floor(share).astype(int)
alloc[(share - alloc).sort_values(ascending=False).index[:SAMPLE_SIZE - alloc.sum()]] += 1
sample = pd.concat([g.sample(n=alloc[r], random_state=SEED) for r, g in pop.groupby('relation') if alloc[r] > 0])
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"population: {len(pop):,} tier-B triples | sample: {len(sample)}")
print(alloc[alloc > 0].to_string())

earlier sample files: 1 | excluded triples: 89
population: 1,000 tier-B triples | sample: 90
relation
developed by        33
competes with       22
works for            8
headquartered in     5
founded by           5
outperforms          4
available on         3
invested in          3
manufactured by      2
acquired by          1
supported by         1
announced by         1
licensed under       1
runs on              1


In [13]:
sent_text = sentences.set_index(['doc_id', 'sentence_id'])['sentence_text']
def ctx(doc_id, sid, offset):
  return sent_text.get((doc_id, sid + offset), '')
def support_context(row):
  b, a = ctx(row['doc_id'], row['sentence_id'], -1), ctx(row['doc_id'], row['sentence_id'], 1)
  parts = ([f"BEFORE: {b}"] if b else []) + [f"EVIDENCE: {row['sentence_text']}"] + ([f"AFTER: {a}"] if a else [])
  return ' || '.join(parts)
def band(x):
  return '0.50-0.59' if x < 0.6 else '0.60-0.74' if x < 0.75 else '0.75-1.00'

ok_by_triple = ok.groupby(['head_id', 'relation', 'tail_id'])
rows = []
for i, e in sample.iterrows():
  sents = ok_by_triple.get_group((e['head_id'], e['relation'], e['tail_id'])).drop_duplicates('sentence_key')
  order = rng.permutation(len(sents))
  primary, support = sents.iloc[order[0]], [sents.iloc[k] for k in order[1:3]]
  notes = []
  for side in ('head', 'tail'):
    merged = sorted(set(e[f'orig_{side}_ids'].split(' | ')) - {e[f'{side}_id']})
    if merged:
      notes.append(f"{side} merged from {', '.join(merged)}")
  row = {
    'sample_id': f'C{i + 1:03d}',
    'relation_id': 'rel3_' + hashlib.sha1(f"{e['head_id']}|{e['relation']}|{e['tail_id']}".encode()).hexdigest()[:16],
    'reviewer_assignment': f"Reviewer_{'ABC'[i // 30]}",
    'head': e['head'], 'head_type': e['head_type'], 'predicate': e['relation'], 'tail': e['tail'], 'tail_type': e['tail_type'],
    'evidence_score': primary['score'], 'mean_score': e['mean_score'], 'score_band': band(e['mean_score']),
    'n_sentences': int(e['n_distinct_sentences']), 'n_docs': int(e['n_docs']), 'tier': e['tier'],
    'correction_notes': '; '.join(notes),
    'primary_context_before': ctx(primary['doc_id'], primary['sentence_id'], -1),
    'evidence_text': primary['sentence_text'],
    'primary_context_after': ctx(primary['doc_id'], primary['sentence_id'], 1),
    'support_2_context': support_context(support[0]) if len(support) > 0 else '',
    'support_3_context': support_context(support[1]) if len(support) > 1 else '',
    'review_correct': '', 'reviewer': '', 'review_error_type': '', 'review_notes': '',
    'head_id': e['head_id'], 'tail_id': e['tail_id'], 'document_id': primary['doc_id'],
    'text_unit_id': f"{primary['doc_id']}:{primary['sentence_id']}", 'published_at': primary['date'],
    'quarter': primary['quarter'], 'extraction_method': 'glirel+wp3_correction', 'selection_reason': 'proportional_by_relation',
    'original_head_ids': e['orig_head_ids'], 'original_tail_ids': e['orig_tail_ids'],
  }
  for k, s in enumerate(support, start=2):
    row[f'support_{k}_document_id'] = s['doc_id']
    row[f'support_{k}_text_unit_id'] = f"{s['doc_id']}:{s['sentence_id']}"
    row[f'support_{k}_score'] = s['score']
  rows.append(row)
review = pd.DataFrame(rows)
for k in (2, 3):
  for c in ('document_id', 'text_unit_id', 'score'):
    if f'support_{k}_{c}' not in review:
      review[f'support_{k}_{c}'] = ''
display(review[['sample_id', 'head', 'predicate', 'tail', 'n_sentences', 'correction_notes']].head(10))

,sample_id,head,predicate,tail,n_sentences,correction_notes
0,C001,Kimi,competes with,DeepSeek,11,
1,C002,R1 reasoning model,developed by,DeepSeek,6,
2,C003,Anthropic,competes with,DeepSeek,26,
3,C004,Dell,competes with,Lenovo,2,
4,C005,Kimi,competes with,Qwen,5,
5,C006,Amit Zavery,works for,ServiceNow,2,
6,C007,MoonShot AI,headquartered in,Beijing,2,head merged from org:moonshot
7,C008,Chrome DevTools,supported by,Bun,2,
8,C009,DeepSeek,headquartered in,Beijing,2,
9,C010,AMD,competes with,Broadcom,3,


In [14]:
REVIEW_NAME = f'relation_review_corrected_seed{SEED}'
review_csv = EVAL_DIR / f'{REVIEW_NAME}.csv'
if review_csv.exists():
  # The sample is frozen once it has been handed to reviewers: never overwrite it (it holds their labels)
  print(f'{review_csv.name} already exists; the distributed review sample is left unchanged')
else:
  review.to_csv(review_csv, index=False)

  # Excel copy with dropdowns (same layout as the earlier review workbook)
  from openpyxl.utils import get_column_letter
  from openpyxl.worksheet.datavalidation import DataValidation
  ERROR_TYPES = ['entity_boundary', 'entity_linking', 'wrong_predicate', 'reversed_direction', 'negation_or_uncertainty',
                 'insufficient_context', 'duplicate_or_noise', 'schema_or_catalogue', 'other']
  guide = pd.DataFrame([
    ('Relation', 'Judge whether head, predicate and tail are supported by the supplied contexts, including direction.'),
    ('Primary context', 'Read the sentence before, the extracted evidence sentence and the sentence after.'),
    ('Additional evidence', 'Use support_2_context and support_3_context when the primary sentence is weak or unclear.'),
    ('correction_notes', 'Entities merged by the WP3 correction (aliases, name variants). Judge the corrected entity.'),
    ('TRUE', 'The relation and direction are supported by the supplied context.'),
    ('FALSE', 'The relation is wrong, reversed, noisy or links the wrong entities.'),
    ('AMBIGUOUS', 'The supplied contexts are still insufficient to decide.'),
    ('reviewer', 'Enter your name or agreed team ID for every labeled row.'),
    ('review_error_type', 'Required for FALSE rows. Choose one of the dropdown values.'),
  ], columns=['Field', 'Instruction'])
  with pd.ExcelWriter(EVAL_DIR / f'{REVIEW_NAME}.xlsx', engine='openpyxl') as xw:
    review.to_excel(xw, sheet_name='Review', index=False)
    guide.to_excel(xw, sheet_name='Guide', index=False)
    ws = xw.sheets['Review']
    ws.freeze_panes = 'D2'
    col = {c: get_column_letter(i + 1) for i, c in enumerate(review.columns)}
    last = len(review) + 1
    for field, values in (('review_correct', ['TRUE', 'FALSE', 'AMBIGUOUS']), ('review_error_type', ERROR_TYPES)):
      dv = DataValidation(type='list', formula1='"' + ','.join(values) + '"', allow_blank=True)
      dv.add(f"{col[field]}2:{col[field]}{last}")
      ws.add_data_validation(dv)
    xw.sheets['Guide'].column_dimensions['B'].width = 110

  sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
  manifest = {
    'version': 'wp3-corrected-review-sample-v1',
    'generated_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'status': 'pending_human_review',
    'sampling_unit': 'unique corrected (head_id, relation, tail_id) GLiREL triple',
    'population': 'tier B: type-valid after correction, no generic node, no lexicon-developer conflict, '
                  'no own-product competition, no reverse duplicate, >= 2 distinct evidence sentences',
    'allocation': 'proportional by relation type (largest remainder), simple random within relation type',
    'random_seed': SEED,
    'excluded_prior_samples': [p.name for p in PRIOR_SAMPLE_FILES if p.exists()],
    'counts': {'tier_counts': edges_corr['tier'].value_counts().sort_index().to_dict(),
               'candidate_graph_unique_edges': int(len(graph_edges)),
               'candidate_graph_duplicate_rows_collapsed': int(len(core) - len(graph_edges)),
               'population_after_prior_exclusion': int(len(pop)), 'sample_rows': int(len(review)),
               'by_relation': review['predicate'].value_counts().to_dict(),
               'by_score_band': review['score_band'].value_counts().sort_index().to_dict()},
    'input_files': {n: sha(WP2_DIR / n) for n in ('nodes.csv', 'edges_relations.csv', 'relations_evidence.csv', 'schema.json')},
    'outputs': {'review_csv': review_csv.name, 'review_csv_sha256': sha(review_csv)},
  }
  (EVAL_DIR / f'{REVIEW_NAME}_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
  print(json.dumps(manifest['counts'], indent=2))

relation_review_corrected_seed2026.csv already exists; the distributed review sample is left unchanged


## Next steps
1. **Review the correction tables** (`artifacts/stage1_wp3/corrections/`): especially `generic_nodes.csv` and the
   curated aliases. Rerun the notebook after each change.
2. **Human review** of `evaluation/relation_review_corrected_seed2026.*` (30 rows per reviewer). The earlier
   summary script can be reused; report precision for tier B and keep the AI-drafted labels out of the evaluation.
3. **Feed back to WP2** for the next stage: add a negative lookahead to the Llama regex
   (`Llama(?![\w.\- ]{0,15}(?:Nemotron|Stack))`), alias normalisation, and sentence de-duplication.
4. Build the graph from `edges_graph_candidates.csv` (deduplicated tiers A + B); keep tier C and the co-occurrence edges (`edges_cooccurrence.csv`) as
   separate, weaker edge types.

**AI Assistance Disclosure**: Claude Code was used to draft this notebook and the seed correction tables.